# STAGE 2: UNIFIED GRAPH CONSTRUCTION

## Goal
Convert UML diagrams into a single unified directed graph.

## Algorithm: Unified Graph Construction
**Input:** Parsed UML elements  
**Output:** Unified Behavioral Graph G

1. Initialize directed graph G
2. For each class: add node in G
3. For each usecase: add node in G
4. For each activity: add node in G
5. For each relationship:
   - include → add edge
   - extend → add edge
   - dependency → add edge
   - control flow → add edge
6. Return graph G

**Graph:** G = (V, E)  
**V** = UML elements  
**E** = relationships

**Complexity:** O(V + E)

In [ ]:
# Install required packages
!pip install networkx matplotlib

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Tuple, Optional
import json

## Define Data Structures

In [ ]:
class NodeType(Enum):
    """Types of nodes in the unified graph"""
    ACTOR = "actor"
    USECASE = "usecase"
    CLASS = "class"
    ACTIVITY = "activity"


class EdgeType(Enum):
    """Types of relationships/edges in the unified graph"""
    INCLUDE = "include"          # Use case includes another
    EXTEND = "extend"            # Use case extends another
    DEPENDENCY = "dependency"     # General dependency
    CONTROL_FLOW = "control_flow" # Sequential flow
    ASSOCIATION = "association"   # General association
    INVOKES = "invokes"          # Actor invokes use case
    USES = "uses"                # Use case uses class/system


@dataclass
class UMLElements:
    """Container for parsed UML elements"""
    actors: List[Tuple[str, str]] = field(default_factory=list)
    usecases: List[Tuple[str, str]] = field(default_factory=list)
    classes: List[Tuple[str, str]] = field(default_factory=list)
    activities: List[Tuple[str, str]] = field(default_factory=list)
    
    # Relationships with types
    includes: List[Tuple[str, str]] = field(default_factory=list)
    extends: List[Tuple[str, str]] = field(default_factory=list)
    dependencies: List[Tuple[str, str]] = field(default_factory=list)
    control_flows: List[Tuple[str, str]] = field(default_factory=list)
    associations: List[Tuple[str, str]] = field(default_factory=list)

print("✓ Data structures defined")

## Unified Graph Builder Class

In [ ]:
class UnifiedGraphBuilder:
    """
    Builds a unified behavioral graph from UML elements
    Complexity: O(V + E) where V = nodes, E = edges
    """
    
    def __init__(self):
        self.graph = nx.DiGraph()
        self.nodes = {}
        
    def build_unified_graph(self, uml_elements: UMLElements) -> nx.DiGraph:
        """
        Main algorithm to build unified graph
        """
        # Step 1: Initialize directed graph G
        self.graph = nx.DiGraph()
        self.nodes.clear()
        
        print("Building Unified Graph...")
        print("-" * 60)
        
        # Step 2: Add class nodes
        print(f"Step 2: Adding {len(uml_elements.classes)} class nodes...")
        for class_id, class_name in uml_elements.classes:
            self._add_node(class_id, class_name, NodeType.CLASS)
        
        # Step 3: Add use case nodes
        print(f"Step 3: Adding {len(uml_elements.usecases)} use case nodes...")
        for uc_id, uc_name in uml_elements.usecases:
            self._add_node(uc_id, uc_name, NodeType.USECASE)
        
        # Step 4: Add activity nodes
        print(f"Step 4: Adding {len(uml_elements.activities)} activity nodes...")
        for act_id, act_name in uml_elements.activities:
            self._add_node(act_id, act_name, NodeType.ACTIVITY)
        
        # Step 4b: Add actor nodes
        print(f"Step 4b: Adding {len(uml_elements.actors)} actor nodes...")
        for actor_id, actor_name in uml_elements.actors:
            self._add_node(actor_id, actor_name, NodeType.ACTOR)
        
        # Step 5: Add relationships
        print(f"Step 5: Adding relationships...")
        
        print(f"  - {len(uml_elements.includes)} include edges")
        for src, tgt in uml_elements.includes:
            self._add_edge(src, tgt, EdgeType.INCLUDE)
        
        print(f"  - {len(uml_elements.extends)} extend edges")
        for src, tgt in uml_elements.extends:
            self._add_edge(src, tgt, EdgeType.EXTEND)
        
        print(f"  - {len(uml_elements.dependencies)} dependency edges")
        for src, tgt in uml_elements.dependencies:
            self._add_edge(src, tgt, EdgeType.DEPENDENCY)
        
        print(f"  - {len(uml_elements.control_flows)} control flow edges")
        for src, tgt in uml_elements.control_flows:
            self._add_edge(src, tgt, EdgeType.CONTROL_FLOW)
        
        print(f"  - {len(uml_elements.associations)} association edges")
        for src, tgt in uml_elements.associations:
            self._add_edge(src, tgt, EdgeType.ASSOCIATION)
        
        # Step 6: Return unified graph
        print("-" * 60)
        print(f"✓ Unified graph constructed:")
        print(f"  Total nodes (V): {self.graph.number_of_nodes()}")
        print(f"  Total edges (E): {self.graph.number_of_edges()}")
        print(f"  Complexity: O({self.graph.number_of_nodes()} + {self.graph.number_of_edges()}) = O({self.graph.number_of_nodes() + self.graph.number_of_edges()})")
        
        return self.graph
    
    def _add_node(self, node_id: str, name: str, node_type: NodeType):
        """Add a node to the unified graph"""
        if node_id not in self.nodes:
            self.nodes[node_id] = {
                'name': name,
                'type': node_type.value
            }
            self.graph.add_node(
                node_id,
                name=name,
                node_type=node_type.value,
                label=name
            )
    
    def _add_edge(self, source: str, target: str, edge_type: EdgeType):
        """Add an edge to the unified graph"""
        if source in self.nodes and target in self.nodes:
            self.graph.add_edge(
                source,
                target,
                edge_type=edge_type.value,
                label=edge_type.value
            )
    
    def get_statistics(self) -> Dict:
        """Get statistics about the unified graph"""
        stats = {
            'total_nodes': self.graph.number_of_nodes(),
            'total_edges': self.graph.number_of_edges(),
            'node_types': {},
            'edge_types': {},
            'density': nx.density(self.graph),
            'is_dag': nx.is_directed_acyclic_graph(self.graph)
        }
        
        # Count nodes by type
        for node_id, data in self.graph.nodes(data=True):
            node_type = data.get('node_type', 'unknown')
            stats['node_types'][node_type] = stats['node_types'].get(node_type, 0) + 1
        
        # Count edges by type
        for src, tgt, data in self.graph.edges(data=True):
            edge_type = data.get('edge_type', 'unknown')
            stats['edge_types'][edge_type] = stats['edge_types'].get(edge_type, 0) + 1
        
        return stats

print("✓ UnifiedGraphBuilder class defined")

## Visualization Function

In [ ]:
def visualize_unified_graph(graph: nx.DiGraph, title="Unified Behavioral Graph G = (V, E)"):
    """Visualize the unified graph"""
    plt.figure(figsize=(14, 10))
    
    # Create layout
    pos = nx.spring_layout(graph, k=2, iterations=50, seed=42)
    
    # Color nodes by type
    node_colors = []
    color_map = {
        'actor': '#FF6B6B',
        'usecase': '#4ECDC4',
        'class': '#45B7D1',
        'activity': '#FFA07A'
    }
    
    for node in graph.nodes():
        node_type = graph.nodes[node].get('node_type', 'unknown')
        node_colors.append(color_map.get(node_type, '#95A5A6'))
    
    # Draw nodes
    nx.draw_networkx_nodes(
        graph, pos,
        node_color=node_colors,
        node_size=1200,
        alpha=0.9
    )
    
    # Draw edges
    edge_colors = []
    for src, tgt in graph.edges():
        edge_type = graph[src][tgt].get('edge_type', 'unknown')
        
        if edge_type == 'include':
            edge_colors.append('#2ECC71')
        elif edge_type == 'extend':
            edge_colors.append('#E74C3C')
        elif edge_type == 'control_flow':
            edge_colors.append('#3498DB')
        else:
            edge_colors.append('#95A5A6')
    
    nx.draw_networkx_edges(
        graph, pos,
        edge_color=edge_colors,
        width=2.5,
        alpha=0.6,
        arrows=True,
        arrowsize=20,
        arrowstyle='->'
    )
    
    # Draw labels
    labels = {node: graph.nodes[node].get('name', node) for node in graph.nodes()}
    nx.draw_networkx_labels(
        graph, pos,
        labels,
        font_size=9,
        font_weight='bold'
    )
    
    # Create legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', 
              markerfacecolor='#FF6B6B', markersize=12, label='Actor'),
        Line2D([0], [0], marker='o', color='w', 
              markerfacecolor='#4ECDC4', markersize=12, label='UseCase'),
        Line2D([0], [0], marker='o', color='w', 
              markerfacecolor='#45B7D1', markersize=12, label='Class'),
        Line2D([0], [0], marker='o', color='w', 
              markerfacecolor='#FFA07A', markersize=12, label='Activity'),
        Line2D([0], [0], color='#2ECC71', linewidth=2, label='Include'),
        Line2D([0], [0], color='#E74C3C', linewidth=2, label='Extend'),
        Line2D([0], [0], color='#3498DB', linewidth=2, label='Control Flow'),
    ]
    plt.legend(handles=legend_elements, loc='upper left', 
              bbox_to_anchor=(1, 1), fontsize=10)
    
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("✓ Visualization function defined")

## Example: Build Unified Graph

In [ ]:
# Create sample UML elements
uml_elements = UMLElements(
    actors=[
        ('A1', 'User'),
        ('A2', 'Admin')
    ],
    usecases=[
        ('UC1', 'Login'),
        ('UC2', 'ValidateCredentials'),
        ('UC3', 'CheckPermissions'),
        ('UC4', 'AccessSystem'),
        ('UC5', 'ManageUsers')
    ],
    classes=[
        ('C1', 'AuthService'),
        ('C2', 'UserDatabase'),
        ('C3', 'PermissionManager')
    ],
    activities=[
        ('ACT1', 'EnterCredentials'),
        ('ACT2', 'SubmitForm'),
        ('ACT3', 'ProcessRequest')
    ]
)

# Define relationships
uml_elements.includes = [
    ('UC1', 'UC2'),  # Login includes ValidateCredentials
    ('UC4', 'UC3'),  # AccessSystem includes CheckPermissions
]

uml_elements.extends = [
    ('UC1', 'UC5'),  # Login extends to ManageUsers (for admin)
]

uml_elements.dependencies = [
    ('UC2', 'C1'),  # ValidateCredentials depends on AuthService
    ('UC2', 'C2'),  # ValidateCredentials depends on UserDatabase
    ('UC3', 'C3'),  # CheckPermissions depends on PermissionManager
]

uml_elements.control_flows = [
    ('ACT1', 'ACT2'),  # EnterCredentials → SubmitForm
    ('ACT2', 'ACT3'),  # SubmitForm → ProcessRequest
]

uml_elements.associations = [
    ('A1', 'UC1'),  # User invokes Login
    ('A2', 'UC5'),  # Admin invokes ManageUsers
    ('UC1', 'UC4'),  # Login leads to AccessSystem
]

print("✓ Sample UML elements created")

In [ ]:
# Build unified graph
builder = UnifiedGraphBuilder()
unified_graph = builder.build_unified_graph(uml_elements)

## Display Statistics

In [ ]:
# Get statistics
stats = builder.get_statistics()

print("\n" + "="*60)
print("UNIFIED GRAPH STATISTICS")
print("="*60)
print(f"\nGraph Structure:")
print(f"  G = (V, E)")
print(f"  |V| = {stats['total_nodes']} nodes")
print(f"  |E| = {stats['total_edges']} edges")
print(f"\nGraph Properties:")
print(f"  Density: {stats['density']:.4f}")
print(f"  Is DAG: {stats['is_dag']}")
print(f"\nNode Distribution:")
for node_type, count in stats['node_types'].items():
    print(f"  {node_type}: {count}")
print(f"\nEdge Distribution:")
for edge_type, count in stats['edge_types'].items():
    print(f"  {edge_type}: {count}")
print(f"\nComplexity Analysis:")
print(f"  Time: O(V + E) = O({stats['total_nodes']} + {stats['total_edges']}) = O({stats['total_nodes'] + stats['total_edges']})")
print(f"  Space: O(V + E) = O({stats['total_nodes'] + stats['total_edges']})")
print("="*60)

## Visualize Unified Graph

In [ ]:
# Visualize the unified graph
visualize_unified_graph(unified_graph)

## Export Graph

In [ ]:
# Export as adjacency list
print("\nAdjacency List Representation:")
print("-" * 60)
for node in unified_graph.nodes():
    neighbors = list(unified_graph.neighbors(node))
    node_name = unified_graph.nodes[node]['name']
    if neighbors:
        neighbor_names = [unified_graph.nodes[n]['name'] for n in neighbors]
        print(f"{node_name} → {', '.join(neighbor_names)}")
    else:
        print(f"{node_name} → (no outgoing edges)")

In [ ]:
# Export to JSON
graph_data = {
    'nodes': [
        {
            'id': node,
            'name': unified_graph.nodes[node]['name'],
            'type': unified_graph.nodes[node]['node_type']
        }
        for node in unified_graph.nodes()
    ],
    'edges': [
        {
            'source': src,
            'target': tgt,
            'type': unified_graph[src][tgt]['edge_type']
        }
        for src, tgt in unified_graph.edges()
    ]
}

print("\nJSON Representation:")
print(json.dumps(graph_data, indent=2))

## Summary

### Algorithm Verification

✅ **Step 1:** Initialize directed graph G  
✅ **Step 2:** Add all class nodes to G  
✅ **Step 3:** Add all usecase nodes to G  
✅ **Step 4:** Add all activity nodes to G  
✅ **Step 5:** Add all relationships as edges:  
   - Include relationships
   - Extend relationships
   - Dependency relationships
   - Control flow relationships
   - Association relationships

✅ **Step 6:** Return unified graph G = (V, E)

### Complexity Analysis

- **Time Complexity:** O(V + E)
  - Adding V nodes: O(V)
  - Adding E edges: O(E)
  - Total: O(V + E)

- **Space Complexity:** O(V + E)
  - Storing V nodes: O(V)
  - Storing E edges: O(E)
  - Total: O(V + E)

### Output

The unified behavioral graph G = (V, E) where:
- **V** = Set of all UML elements (actors, use cases, classes, activities)
- **E** = Set of all relationships (include, extend, dependency, control flow, association)